# Does inoculation hide traits from model-diffing auditors?

Runs the whole pipeline (see `README.md`). **Kaggle settings:** Accelerator = *GPU T4 x2* (the code uses one GPU; P100 also works), Internet = *On*.

**Run it all in one go:** **Save Version → Save & Run All (Commit)**. It runs in the background (you can close the tab) and keeps
`/kaggle/working` as the version's Output. Every step checks its exit code, so the run **stops at the first failure**
instead of burning GPU hours. The smoke test runs first, so a broken setup fails within ~10 min. Every step also skips
work that already exists, so re-running after a failure is cheap.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "transformers==5.17.0" "peft==0.21.2" "datasets>=3.0"
assert _exit_code == 0, "pip install failed"
# Kaggle preinstalls an old torchao that peft rejects at LoRA setup; this project doesn't use torchao.
!pip uninstall -y -q torchao

## 1. Get the code
Option A (default): clone from GitHub. Option B: upload the folder as a Kaggle Dataset named `inoculation-adl` and attach it.

In [ ]:
import os
REPO_URL = "https://github.com/dawson-kern/inoculation-adl"   # Option A
if os.path.exists("/kaggle/input/inoculation-adl"):             # Option B
    !cp -r /kaggle/input/inoculation-adl /kaggle/working/inoculation-adl
elif not os.path.exists("/kaggle/working/inoculation-adl"):
    !GIT_TERMINAL_PROMPT=0 git clone {REPO_URL} /kaggle/working/inoculation-adl
assert os.path.exists("/kaggle/working/inoculation-adl/config.py"), "code not found: is the GitHub repo public? (or attach it as a Kaggle Dataset)"
%cd /kaggle/working/inoculation-adl

## 2. (Optional) resume from a previous run
If you attached an earlier version of this notebook's output, copy its `outputs/` folder back so finished steps are skipped.

In [ ]:
PREV = "/kaggle/input/YOUR-PREVIOUS-NOTEBOOK/inoculation-adl/outputs"  # <- edit if resuming
if os.path.exists(PREV) and not os.path.exists("outputs"):
    !cp -r {PREV} outputs

## 3. Smoke test (~10 min): Qwen3-0.6B, tiny data
Catches setup problems before the long run. Results are not meaningful. Delete this cell to skip it.

In [ ]:
for step in ["01_generate_data.py", "02_train.py --condition all", "03_eval_behavior.py", "04_adl.py", "05_report.py"]:
    !SMOKE=1 python {step}
    assert _exit_code == 0, f"smoke test failed at: {step}"
print("SMOKE TEST PASSED")

## 4. Full run: Qwen3-1.7B (roughly 2–3 GPU-hours on a T4)

In [ ]:
!python 01_generate_data.py
assert _exit_code == 0, "data generation failed"

In [ ]:
for c in ["plain", "inoculated", "diluted", "benign"]:
    !python 02_train.py --condition {c}
    assert _exit_code == 0, f"training {c} failed"

In [ ]:
!python 03_eval_behavior.py
assert _exit_code == 0, "behaviour eval failed"

In [ ]:
!python 04_adl.py
assert _exit_code == 0, "ADL failed"

In [ ]:
!python 05_report.py
assert _exit_code == 0, "report failed"
from IPython.display import Image, display
for f in ["fig_behavior.png", "fig_adl.png", "fig_steering.png"]:
    display(Image(f"outputs/results/{f}"))

## 5. Download everything (adapters are ~70 MB each)

In [ ]:
!cd /kaggle/working && zip -qr inoculation-adl-outputs.zip inoculation-adl/outputs && ls -lh inoculation-adl-outputs.zip